# Rubic-RFL — Train RL model on Colab (cloud GPU)

**How to use from VS Code:** open this notebook, then in the kernel picker (top-right) connect to a **Colab** runtime with a **GPU**. Every cell below runs on Google's VM, *not* your laptop — that's why step 1 clones the repo.

Run the cells top to bottom. Outputs are written to Google Drive so they survive a disconnect.

## 1. Confirm we're on a GPU runtime

In [1]:
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime to GPU before continuing'

GPU 0: Tesla T4 (UUID: GPU-e5d31cdf-b8e0-dcaf-9701-3889e91eabe7)
CUDA available: True


## 2. Clone the repo (runs on the remote VM)
If the repo is **private**, replace the URL with a token form:
`https://<github-username>:<personal-access-token>@github.com/IricsDo/Rubic-RFL.git`

In [ ]:
%cd /content
# DAVI code lives on the feature/davi branch.
![ -d Rubic-RFL ] && (cd Rubic-RFL && git fetch origin && git checkout feature/davi && git pull) || git clone --branch feature/davi https://github.com/IricsDo/Rubic-RFL.git
%cd /content/Rubic-RFL

## 3. Install BOTH packages
The trainer does `from app.cube import Cube`, so the **backend** package is required — not just `rl`. Torch is already on Colab with the right CUDA build, so we don't reinstall it.

**Note:** `pip install -e` writes a `.pth` file that the *already-running* kernel won't re-read, so we also add the package roots to `sys.path` to make them importable in this kernel **without a restart**. (The `!python -m ...` training cells spawn fresh processes and pick up the editable install on their own.)

In [3]:
!pip install -q -e backend   # provides app.cube  (required by the trainer)
!pip install -q -e rl        # provides rubic_rl

# Make both packages importable in THIS running kernel without a restart:
import sys
for p in ('/content/Rubic-RFL/backend', '/content/Rubic-RFL/rl'):
    if p not in sys.path:
        sys.path.insert(0, p)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.4 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 144.8 MB/s eta 0:00:0000:01
  Building editable for rubic-rfl-backend (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rubic-rfl-rl (pyproject.toml) ... done


In [4]:
from app.cube import Cube
from rubic_rl.training import adi
print('cross-package imports OK')

cross-package imports OK


## 4. Mount Google Drive (so checkpoints survive disconnects)

In [5]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rubic-rfl'
import os
for sub in ('datasets', 'checkpoints', 'reports'):
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)
print('Saving outputs under', OUT)

Mounted at /content/drive
Saving outputs under /content/drive/MyDrive/rubic-rfl


## 5. Smoke test (~1 min) — proves the pipeline runs before a long job

In [ ]:
%cd /content/Rubic-RFL/rl
!python -m rubic_rl.training.adi \
  --depths 1 2 3 --samples-per-depth 100 \
  --iterations 1 --epochs-per-iteration 5 --device cuda \
  --dataset-out /content/drive/MyDrive/rubic-rfl/datasets/adi-smoke.jsonl \
  --checkpoint-out /content/drive/MyDrive/rubic-rfl/checkpoints/torch-policy-value-adi-smoke.pt \
  --report-out /content/drive/MyDrive/rubic-rfl/reports/adi-training-smoke.json

## 6. Full training — depth 1–30 curriculum
This is the long run. Lower `--iterations` or `--samples-per-depth` if you hit free-tier time limits.

In [6]:
%cd /content/Rubic-RFL/rl
# Stronger run, memory-safe for free Colab (~12.7GB RAM).
# --train-on-latest-records bounds the per-iteration feature matrix to one iteration's data
# (~180k rows) instead of the growing cumulative replay, so RAM stays flat across iterations.
# This keeps high samples-per-depth + many iterations without the host-RAM OOM that killed the
# cumulative run. New -v2 outputs keep the baseline intact. NOTE: no resume — must finish in one session.
!python -m rubic_rl.training.adi \
  --depths 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30 \
  --samples-per-depth 6000 --canonical-depths 1 2 \
  --iterations 10 --epochs-per-iteration 6 \
  --train-on-latest-records \
  --hidden-dim 512 --residual-blocks 4 --dropout 0.1 \
  --batch-size 2048 --learning-rate 0.0003 --weight-decay 0.001 \
  --value-loss-weight 0.25 --seed 20260603 --device cuda \
  --model-version torch-policy-value-v0.2 \
  --dataset-out /content/drive/MyDrive/rubic-rfl/datasets/adi-depth-1-30-curriculum-v2.jsonl \
  --checkpoint-out /content/drive/MyDrive/rubic-rfl/checkpoints/torch-policy-value-adi-depth-1-30-v2.pt \
  --report-out /content/drive/MyDrive/rubic-rfl/reports/adi-training-depth-1-30-v2.json

/content/Rubic-RFL/rl
{
  "dataset": {
    "canonical_depths": [
      1,
      2
    ],
    "depths": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30
    ],
    "include_solved": true,
    "path": "/content/drive/MyDrive/rubic-rfl/datasets/adi-depth-1-30-curriculum-v2.jsonl",
    "records": 1584263,
    "samples_per_depth": 6000,
    "seed": 20260603,
    "trainable_policy_records": 1584262,
    "value_scale": 30.0
  },
  "experiment": "adi-policy-value",
  "model": {
    "checkpoint_out": "/content/drive/MyDrive/rubic-rfl/checkpoints/torch-policy-value-adi-depth-1-30-v2.pt",
    "config": {
      "action_count": 18,
      "dropout": 0.1,
      "hidden_dim": 512,
      "input_size": 324,
      "residual_blocks": 4
    },
    "metadata": {
      

## 7. Evaluate — the UI promotion gate
Check `summary.overall.solve_rate` and `summary.by_depth` in the report. That's your go/no-go before using the model in the UI.

In [7]:
!python -m rubic_rl.evaluation.search_eval \
  --model /content/drive/MyDrive/rubic-rfl/checkpoints/torch-policy-value-adi-depth-1-30-v2.pt \
  --policy-type auto \
  --depths 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30 \
  --samples-per-depth 50 --max-depth 30 --beam-width 10 --top-k 8 --seed 20260603 \
  --out /content/drive/MyDrive/rubic-rfl/reports/adi-search-evaluation-depth-1-30-v2.json

import json
report = json.load(open('/content/drive/MyDrive/rubic-rfl/reports/adi-search-evaluation-depth-1-30-v2.json'))

# The report holds two strategies (greedy + beam_search); the summary lives inside each.
for strat in report['strategies']:
    o = strat['summary']['overall']
    print(f"{strat['label']:12s} solve_rate={o['solve_rate']:.3f}  "
          f"avg_moves={o['avg_move_count_solved']}  avg_ms={o['avg_duration_ms']}")

# beam_search is the UI promotion gate — print its per-depth solve rate:
beam = next(s for s in report['strategies'] if s['label'] == 'beam_search')
print('\nbeam_search by_depth solve_rate:')
for depth, stats in sorted(beam['summary']['by_depth'].items(), key=lambda kv: int(kv[0])):
    print(f"  depth {depth:>2}: {stats['solve_rate']:.3f}")

greedy       solve_rate=0.197  avg_moves=3.6621621621621623  avg_ms=30.67528612200037
beam_search  solve_rate=0.259  avg_moves=4.788659793814433  avg_ms=335.8700002613335

beam_search by_depth solve_rate:
  depth  1: 1.000
  depth  2: 1.000
  depth  3: 1.000
  depth  4: 1.000
  depth  5: 1.000
  depth  6: 0.920
  depth  7: 0.820
  depth  8: 0.420
  depth  9: 0.420
  depth 10: 0.120
  depth 11: 0.040
  depth 12: 0.000
  depth 13: 0.000
  depth 14: 0.020
  depth 15: 0.000
  depth 16: 0.000
  depth 17: 0.000
  depth 18: 0.000
  depth 19: 0.000
  depth 20: 0.000
  depth 21: 0.000
  depth 22: 0.000
  depth 23: 0.000
  depth 24: 0.000
  depth 25: 0.000
  depth 26: 0.000
  depth 27: 0.000
  depth 28: 0.000
  depth 29: 0.000
  depth 30: 0.000


In [8]:
# ── WIDE-BEAM DIAGNOSTIC ───────────────────────────────────────────────────
# Is the deep-depth wall search-limited or policy-limited?
# Re-evaluate v2 on the contested band (depths 7-14) with a 10x wider beam.
#   - If solve rates JUMP  -> search-limited: just spend more search at inference.
#   - If solve rates are FLAT -> policy-limited: need a bigger net / value-guided search.
# Same checkpoint, same seed, same samples-per-depth (50) as the beam=10 run -> directly comparable.
# Heads-up: beam=100 is ~10x slower per solve; this band may take ~20-30 min.
%cd /content/Rubic-RFL/rl
!python -m rubic_rl.evaluation.search_eval \
  --model /content/drive/MyDrive/rubic-rfl/checkpoints/torch-policy-value-adi-depth-1-30-v2.pt \
  --policy-type auto \
  --depths 7,8,9,10,11,12,13,14 \
  --samples-per-depth 50 --max-depth 30 --beam-width 100 --top-k 12 --seed 20260603 \
  --out /content/drive/MyDrive/rubic-rfl/reports/adi-search-eval-v2-widebeam.json

import json
def beam_by_depth(path):
    r = json.load(open(path))
    beam = next(s for s in r['strategies'] if s['label'] == 'beam_search')
    return {int(d): v['solve_rate'] for d, v in beam['summary']['by_depth'].items()}

narrow = beam_by_depth('/content/drive/MyDrive/rubic-rfl/reports/adi-search-evaluation-depth-1-30-v2.json')
wide   = beam_by_depth('/content/drive/MyDrive/rubic-rfl/reports/adi-search-eval-v2-widebeam.json')

print(f"{'depth':>5} {'beam=10':>9} {'beam=100':>9} {'delta':>8}")
for d in sorted(wide):
    n = narrow.get(d, float('nan'))
    print(f"{d:>5} {n:>9.2f} {wide[d]:>9.2f} {wide[d]-n:>+8.2f}")

/content/Rubic-RFL/rl
depth   beam=10  beam=100    delta
    7      0.82      0.98    +0.16
    8      0.42      0.86    +0.44
    9      0.42      0.62    +0.20
   10      0.12      0.38    +0.26
   11      0.04      0.28    +0.24
   12      0.00      0.02    +0.02
   13      0.00      0.20    +0.20
   14      0.02      0.04    +0.02


## 8. Bring the model home
The trained `torch-policy-value-adi-depth-1-30.pt` is now in your Google Drive under `rubic-rfl/checkpoints/`. Download it, drop it into your local `checkpoints/`, then point the backend at it:

```powershell
$env:RUBIC_RL_MODEL_PATH="checkpoints\torch-policy-value-adi-depth-1-30.pt"
$env:RUBIC_RL_POLICY_TYPE="torch"
```

## 9. DAVI — value-iteration để nhắm depth sâu (hướng DeepCubeA)

Khác với policy supervised ở trên (trần ~depth 11), DAVI học hàm giá trị `V(s)` = số nước tới lời giải bằng bootstrapping ngược từ trạng thái solved, rồi giải bằng **Weighted A\***.

**Resume tự động:** file `.resume` nằm trên Drive, nên chạy lại cell train là tiếp tục từ step đã lưu — sống sót qua việc Colab ngắt phiên. Cứ chạy lại nhiều lần để tích lũy compute.

In [ ]:
%cd /content/Rubic-RFL/rl
# DAVI training. Re-run this cell to resume (it reads <checkpoint>.resume on Drive).
# Tip: raise --iterations across sessions to accumulate compute toward deeper solving.
!python -m rubic_rl.training.davi \
  --device cuda \
  --iterations 20000 --batch-size 1000 \
  --hidden-dim 512 --residual-blocks 6 --dropout 0.0 \
  --learning-rate 0.001 --weight-decay 0.00001 \
  --target-update-interval 200 --checkpoint-interval 200 \
  --curriculum-start 1 --curriculum-interval 150 --max-scramble-depth 30 \
  --seed 20260603 --model-version torch-value-davi-v0.1 \
  --checkpoint-out /content/drive/MyDrive/rubic-rfl/checkpoints/torch-value-davi.pt \
  --report-out /content/drive/MyDrive/rubic-rfl/reports/davi-training.json

## 10. Đánh giá DAVI bằng Weighted A* (solve-rate theo depth)

Đây là cổng đánh giá của nhánh value-iteration (tương đương `search_eval` của nhánh policy). Lúc model còn yếu, hãy thu hẹp `--depths` (vd `1,2,...,12`) để chạy nhanh; khi mạnh dần thì mở rộng tới 30.

In [ ]:
%cd /content/Rubic-RFL/rl
!python -m rubic_rl.evaluation.davi_eval \
  --model /content/drive/MyDrive/rubic-rfl/checkpoints/torch-value-davi.pt \
  --depths 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30 \
  --samples-per-depth 50 --weight 0.6 --batch-expansion 1000 --max-nodes 1000000 \
  --device cuda \
  --out /content/drive/MyDrive/rubic-rfl/reports/davi-eval.json

import json
rep = json.load(open('/content/drive/MyDrive/rubic-rfl/reports/davi-eval.json'))
print('overall:', rep['summary']['overall'])
print('\nby_depth solve_rate:')
for d, s in sorted(rep['summary']['by_depth'].items(), key=lambda kv: int(kv[0])):
    print(f"  depth {int(d):>2}: {s['solve_rate']:.3f}  (avg_len={s['avg_len_solved']}, avg_ms={s['avg_ms']:.0f})")